### Imports & Setup
Importing necessary libraries (PyTorch, Transformers, Torchvision, Pandas, etc.) and setting up file paths for the pre-trained models and datasets.


In [1]:
import os, torch, numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
BASE      = ""
CNN_PATH  = "best_model.pth"
BERT_PATH = "best_bert_model.pth"
IMG_DIR   = "cancer_data/train/"
VAL_CSV   = "val_split.csv"
TRAIN_CSV = "train_split.csv"

Device: cuda


### Image Model (CNN)
Defining the ResNet18-based architecture used for analyzing histopathology images and loading its pre-trained weights. We also define a `get_features` method to extract the 512-dimensional vector before the final classification layer.


In [2]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        resnet    = models.resnet18(pretrained=False)
        resnet.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, 2)
        )
        self.model = resnet

    def forward(self, x):
        return self.model(x)

    def get_features(self, x):
        # Returns 512-dim vector before classifier
        x = self.model.conv1(x); x = self.model.bn1(x)
        x = self.model.relu(x);  x = self.model.maxpool(x)
        x = self.model.layer1(x); x = self.model.layer2(x)
        x = self.model.layer3(x); x = self.model.layer4(x)
        x = self.model.avgpool(x)
        return x.view(x.size(0), -1)   # [batch, 512]

cnn_model = CNNModel().to(device)
cnn_model.model.load_state_dict(torch.load(CNN_PATH, map_location=device))
cnn_model.eval()
print("CNN loaded!")

CNN loaded!


### Text Model (BERT)
Defining the ClinicalBERT-based architecture used for analyzing medical discharge summaries and loading its pre-trained weights. We also define a `get_features` method to extract the raw 768-dimensional `[CLS]` embedding.


In [ ]:
class BertCancerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert       = BertModel.from_pretrained("bert-base-uncased")
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  # [CLS] token = 768-dim
        return self.classifier(cls)

    def get_features(self, input_ids, attention_mask):
        # Returns raw 768-dim CLS embedding
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # [batch, 768]

bert_model  = BertCancerClassifier().to(device)
ckpt        = torch.load(BERT_PATH, map_location=device)
bert_model.load_state_dict(ckpt["model_state"])
bert_model.eval()
tokenizer   = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"BERT loaded!  val_acc from training: {ckpt.get('val_acc','?')}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Multimodal Fusion Architecture
Defining the `MultimodalFusion` network. This MLP takes the 512-dim image features from the CNN and the 768-dim text features from BERT, concatenates them into a 1280-dim vector, and passes them through a few dense layers to produce the final classification.


In [ ]:
class MultimodalFusion(nn.Module):
    """
    Takes raw feature vectors from CNN and BERT.

    CNN  → 512-dim image features
    BERT → 768-dim text features

    Concat → 1280-dim → MLP → 2 outputs
    """

    def __init__(self):
        super().__init__()

        self.fusion = nn.Sequential(
            nn.Linear(512 + 768, 512),  # 1280 → 512
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),        # 512 → 128
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 2)           # 128 → cancer/normal
        )

    def forward(self, img_feat, txt_feat):
        # Concatenate image + text features
        combined = torch.cat([img_feat, txt_feat], dim=1)  # [batch, 1280]

        # Pass through fusion MLP
        return self.fusion(combined)  # [batch, 2]


# ---------------------------
# Initialize model
# ---------------------------
fusion_model = MultimodalFusion().to(device)


# ---------------------------
# Parameter count
# ---------------------------
params = sum(p.numel() for p in fusion_model.parameters())
print(f"Fusion model ready! Parameters: {params:,}")

### Multimodal Dataset
Loading and merging the MIMIC-III clinical notes and creating a `FusionDataset`. This custom dataset loads both the image and the corresponding text note for each sample.


In [ ]:
# Load real MIMIC notes with cancer labels
NOTES_PATH = BASE + "NOTEEVENTS.csv"
DIAG_PATH  = BASE + "DIAGNOSES_ICD.csv"
notes_df = pd.read_csv(NOTES_PATH,
               usecols=["SUBJECT_ID", "HADM_ID", "CATEGORY", "TEXT"])

notes_df = notes_df[notes_df["CATEGORY"] == "Discharge summary"]
notes_df = notes_df.dropna(subset=["TEXT"])

diag_df  = pd.read_csv(DIAG_PATH,
               usecols=["HADM_ID", "ICD9_CODE"])

def is_cancer(code):
    try:
        c = str(code).strip()
        if c[0].upper() in ["V","E"]: return 0
        return 1 if 140 <= float(c[:3]) <= 239 else 0
    except: return 0

diag_df["cancer"] = diag_df["ICD9_CODE"].apply(is_cancer)
labels   = diag_df.groupby("HADM_ID")["cancer"].max().reset_index()
labels.columns = ["HADM_ID", "mimic_label"]

mimic_df = notes_df.merge(labels, on="HADM_ID")
mimic_df = mimic_df.drop_duplicates("HADM_ID")
mimic_df["TEXT"] = mimic_df["TEXT"].str[:512]

cancer_notes = mimic_df[mimic_df["mimic_label"]==1]["TEXT"].tolist()
normal_notes = mimic_df[mimic_df["mimic_label"]==0]["TEXT"].tolist()
print(f"Real cancer notes: {len(cancer_notes)}")
print(f"Real normal notes: {len(normal_notes)}")

def get_real_note(label, idx):
    pool = cancer_notes if label == 1 else normal_notes
    return pool[idx % len(pool)]

transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                           [0.229,0.224,0.225])
])

class FusionDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["label"])

        img  = Image.open(IMG_DIR + row["id"] + ".tif").convert("RGB")
        img  = transform(img)

        note = get_real_note(label, idx)
        enc  = tokenizer(note, max_length=256,
                          padding="max_length", truncation=True,
                          return_tensors="pt")
        return {
            "image"         : img,
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(label, dtype=torch.long)
        }

train_ds = FusionDataset(TRAIN_CSV)
val_ds   = FusionDataset(VAL_CSV)

# Use larger batch + more workers — these loaders are only used ONCE for feature extraction
extract_train_loader = DataLoader(train_ds, batch_size=64, shuffle=False,
                                   num_workers=6, pin_memory=True, persistent_workers=True)
extract_val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                                   num_workers=6, pin_memory=True, persistent_workers=True)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

### Feature Extraction & Caching
To drastically speed up training, we run all images through the frozen CNN and all texts through the frozen BERT *once*. We cache these extracted feature vectors into RAM so the fusion MLP can train instantly without I/O or heavy model overhead.


In [ ]:
from tqdm.auto import tqdm
from torch.utils.data import TensorDataset

# ----------------------------------------------------------
# Pre-compute CNN + BERT features ONCE (both are frozen).
# This runs each sample through the heavy models a single
# time; the fusion MLP then trains on the cached tensors,
# keeping the GPU 100% busy with no I/O or BERT overhead.
# ----------------------------------------------------------

def extract_features(loader, desc=""):
    all_img, all_txt, all_lbl = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            images = batch["image"].to(device, non_blocking=True)
            ids    = batch["input_ids"].to(device, non_blocking=True)
            mask   = batch["attention_mask"].to(device, non_blocking=True)

            img_feat = cnn_model.get_features(images)          # [B, 512]
            txt_feat = bert_model.get_features(ids, mask)      # [B, 768]

            all_img.append(img_feat.cpu())
            all_txt.append(txt_feat.cpu())
            all_lbl.append(batch["label"])

    return torch.cat(all_img), torch.cat(all_txt), torch.cat(all_lbl)


print("Extracting train features (runs once)...")
tr_img, tr_txt, tr_lbl = extract_features(extract_train_loader, "Train")

print("Extracting val features (runs once)...")
vl_img, vl_txt, vl_lbl = extract_features(extract_val_loader, "Val")

# Build fast TensorDatasets — no image I/O, no tokenization, no BERT/CNN during training
train_feat_ds = TensorDataset(tr_img, tr_txt, tr_lbl)
val_feat_ds   = TensorDataset(vl_img, vl_txt, vl_lbl)

# Large batch, no workers needed (tensors already in RAM)
train_loader = DataLoader(train_feat_ds, batch_size=256, shuffle=True,
                           num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_feat_ds,   batch_size=256, shuffle=False,
                           num_workers=0, pin_memory=True)

print(f"\nCached features ready — Train: {len(train_feat_ds)}  Val: {len(val_feat_ds)}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Fusion Model Training
Training the `MultimodalFusion` MLP using the cached image and text features. We use mixed precision (`GradScaler`) and a learning rate scheduler to optimize the training process.


In [ ]:
# Freeze CNN and BERT (only fusion MLP has gradients)
for p in cnn_model.parameters():  p.requires_grad = False
for p in bert_model.parameters(): p.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.cuda.amp.GradScaler()

EPOCHS   = 10
best_acc = 0
history  = {"train": [], "val": []}

print("=" * 55)
print(" DeepOncoVision — Fusion Training (cached features)")
print(" CNN: frozen | BERT: frozen | Fusion: training")
print("=" * 55)

for epoch in range(EPOCHS):

    fusion_model.train()
    correct = total = loss_sum = 0

    # train_loader now yields (img_feat, txt_feat, label) tensors directly
    for img_feat, txt_feat, labels in train_loader:
        img_feat = img_feat.to(device, non_blocking=True)
        txt_feat = txt_feat.to(device, non_blocking=True)
        labels   = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = fusion_model(img_feat, txt_feat)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        _, pred = logits.max(1)
        correct  += pred.eq(labels).sum().item()
        total    += labels.size(0)
        loss_sum += loss.item()

    train_acc = 100 * correct / total

    # Validation
    fusion_model.eval()
    vc = vt = 0
    with torch.no_grad():
        for img_feat, txt_feat, labels in val_loader:
            img_feat = img_feat.to(device, non_blocking=True)
            txt_feat = txt_feat.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                logits = fusion_model(img_feat, txt_feat)

            _, pred = logits.max(1)
            vc += pred.eq(labels).sum().item()
            vt += labels.size(0)

    val_acc = 100 * vc / vt
    scheduler.step(val_acc)
    history["train"].append(train_acc)
    history["val"].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(fusion_model.state_dict(), "best_fusion_model.pth")
        print(f" saved best: {best_acc:.1f}%")

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch+1}/{EPOCHS}] train: {train_acc:.1f}%  val: {val_acc:.1f}%  lr: {lr_now}")

print(f"\nBest val accuracy: {best_acc:.1f}%")

### Results & Visualization
Plotting the training/validation accuracy curve for the fusion model and comparing its final performance against the unimodal CNN and BERT baselines.


In [ ]:
# ---------------------------
# Plot setup
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))


# ---------------------------
# Left: training curve
# ---------------------------
axes[0].plot(
    history["train"],
    label="Train",
    color="#4A90D9",
    linewidth=2
)

axes[0].plot(
    history["val"],
    label="Val",
    color="#27AE60",
    linewidth=2
)

axes[0].axhline(
    86,
    color="red",
    linestyle="--",
    label="CNN baseline 86%"
)

axes[0].set_title("Fusion Training Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy %")

axes[0].legend()
axes[0].grid(alpha=0.3)


# ---------------------------
# Right: model comparison bar chart
# ---------------------------
names  = ["CNN\n(Image)", "BERT\n(Text)", "Fusion\n(Combined)"]
accs   = [86.0, 93.0, best_acc]
colors = ["#4A90D9", "#E67E22", "#27AE60"]

bars = axes[1].bar(
    names,
    accs,
    color=colors,
    width=0.5,
    edgecolor="white",
    linewidth=1.5
)

for bar, acc in zip(bars, accs):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{acc:.1f}%",
        ha="center",
        fontweight="bold"
    )

axes[1].set_ylim(70, 100)
axes[1].set_title("DeepOncoVision — Model Comparison")
axes[1].set_ylabel("Accuracy %")
axes[1].grid(axis="y", alpha=0.3)


# ---------------------------
# Finalize plot
# ---------------------------
plt.tight_layout()
plt.savefig("deeponcovision_results.png", dpi=150)
plt.show()


# ---------------------------
# Print results
# ---------------------------
print(f"\nCNN alone: 86.0%")
print(f"BERT alone: 93.0%")
print(f"Fusion: {best_acc:.1f}%")
print(f"Improvement: {best_acc - 86:.1f}% over CNN baseline")